# Nova 1 — phase 1 (code-only LoRA on Qwen/Qwen3.5-4B)

Pushed by the Kaggle monitor on Nova 1's share of the weekly GPU quota. Each session:
1. installs Unsloth (+ transformers v5, needed by Qwen3.5),
2. takes the training mix and the latest checkpoints from the previous session's output (attached as input) or builds the mix (~20M tokens) when there is none,
3. trains on one T4, saving a checkpoint every 50 steps, and stops by itself `MAX_MINUTES` after the session began,
4. leaves `out/` (checkpoints, then the final adapter), `data/mix.jsonl` and `nova1_status.json` in its output for the next session.

In [ ]:
MAX_MINUTES = 600   # set before each push: what is left of Nova 1's weekly GPU share, capped below Kaggle's 12 h session limit
BRANCH = "training/nova1"
REPO = "https://github.com/jonsnow-org/Ttbik"

import glob, json, os, shutil, subprocess, sys, time
T0 = time.time()
WORK = "/kaggle/working"
SRC = "/kaggle/temp/ttbik"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Unsloth trains on one GPU (T4 x2 machine: the second stays idle)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
def sh(cmd, check=True):
    print("+", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r.returncode
def minutes_used():
    return (time.time() - T0) / 60
sh("nvidia-smi", check=False)

In [ ]:
sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {SRC}")
sh("pip install -q --upgrade unsloth unsloth_zoo")
sh('pip install -q --upgrade "transformers>=5.2.0" "trl>=0.24" "peft>=0.17" "datasets>=3.0" datasketch pyyaml')
os.chdir(f"{SRC}/training/nova1")
print("code at", subprocess.run("git -C %s log -1 --format=%%h" % SRC, shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# The previous session's output (this notebook attached as its own input): mix + checkpoints.
prev = sorted(set(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/nova1_status.json", recursive=True)))
prev = prev[0] if prev else None
print("previous output:", prev)
os.makedirs(f"{WORK}/data", exist_ok=True)
os.makedirs(f"{WORK}/out", exist_ok=True)
status = {}
if prev:
    status = json.load(open(f"{prev}/nova1_status.json"))
    if os.path.exists(f"{prev}/data/mix.jsonl"):
        shutil.copy(f"{prev}/data/mix.jsonl", f"{WORK}/data/mix.jsonl")
    for d in glob.glob(f"{prev}/out/checkpoint-*"):
        shutil.copytree(d, f"{WORK}/out/{os.path.basename(d)}", dirs_exist_ok=True)
print("previous status:", status)
if status.get("done"):
    print("training already finished; nothing to do")

In [ ]:
# Build the code-only mix when no previous session left one (a few minutes, streaming from the Hub).
if not status.get("done") and not os.path.exists(f"{WORK}/data/mix.jsonl"):
    sh("timeout 600 python scripts/make_repair_tasks.py --n 6000 --out data/synth_repair.jsonl", check=False)   # at most 10 min of the session (rows are written as they are made)
    sh(f"python scripts/build_mix.py --tokens 20000000 --max-len 2048 --out {WORK}/data/mix.raw.jsonl")
    sh(f"python scripts/dedup_decontam.py {WORK}/data/mix.raw.jsonl {WORK}/data/mix.jsonl")
    os.remove(f"{WORK}/data/mix.raw.jsonl")
os.makedirs("data", exist_ok=True)
if os.path.exists(f"{WORK}/data/mix.jsonl") and not os.path.exists("data/mix.jsonl"):
    os.symlink(f"{WORK}/data/mix.jsonl", "data/mix.jsonl")
sh(f"wc -l {WORK}/data/mix.jsonl", check=False)
print(f"{minutes_used():.0f} min used before training")

In [ ]:
# Train. train.py saves every 50 steps and stops itself when --max-minutes (counted from ITS start) is reached.
rc = 0
if not status.get("done"):
    left = MAX_MINUTES - minutes_used() - 15          # 15 min kept for the last save and the output upload
    if left < 20:
        print("not enough time left in this session for a useful stretch of training")
    else:
        rc = sh(f"python train/train.py --config train/config.kaggle.yaml --out {WORK}/out --resume --max-minutes {left:.0f}", check=False)

In [ ]:
# Status for the next session (and for the monitor): finished when the adapter was saved.
done = os.path.exists(f"{WORK}/out/adapter_config.json")
ckpts = sorted(glob.glob(f"{WORK}/out/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
step = int(ckpts[-1].rsplit("-", 1)[1]) if ckpts else 0
st = {"done": done or bool(status.get("done")), "last_checkpoint_step": step, "train_exit_code": rc,
      "session_minutes": round(minutes_used(), 1), "sessions": int(status.get("sessions", 0)) + 1}
json.dump(st, open(f"{WORK}/nova1_status.json", "w"), indent=1)
print(st)
if rc:
    raise SystemExit(f"training exited with {rc}")